# In-context fine-tune + eval (directed_forgetting -> recent_probes)
Teach M_pop to USE a person's task-A transcript to predict their task-B behavior: fine-tune on [A+B] sequences (loss on B responses), then evaluate real-A vs floor vs shuffled-A.

**GPU:** A100 -> set `MAXLEN = 6144` (full A+B). L4 -> set `MAXLEN = 4096` (B kept, A partly truncated; avoids OOM).

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
if not os.path.exists('/content/sro-minitaur-transfer'):
    !git clone https://github.com/YifeiCAO/sro-minitaur-transfer.git
!cd /content/sro-minitaur-transfer && git pull
%cd /content/sro-minitaur-transfer
!pip -q install -e . && pip -q install -r requirements-model.txt
from huggingface_hub import login; login()  # paste hf_ token

In [ ]:
MAXLEN = 6144   # A100. Change to 4096 if on L4.
SRC, TGT = 'directed_forgetting', 'recent_probes'
print('max_seq_len =', MAXLEN, '|', SRC, '->', TGT)

### 1) Fine-tune on [A+B] sequences -> saves mpop_ic to Drive (~1-2h)

In [ ]:
!cd /content/sro-minitaur-transfer && python scripts/finetune_incontext.py \
    --mpop /content/drive/MyDrive/sro_minitaur/mpop \
    --source {SRC} --target {TGT} \
    --out /content/drive/MyDrive/sro_minitaur/mpop_ic \
    --max-seq-len {MAXLEN} --epochs 2

### 2) Evaluate the fine-tuned model: real-A vs floor vs shuffled-A
`mean_real_minus_floor` < 0 => A context now helps (fine-tune worked). `mean_real_minus_shuffled` < 0 (p<.05) => person-specific transfer.

In [ ]:
!cd /content/sro-minitaur-transfer && python scripts/run_incontext.py \
    --mpop /content/drive/MyDrive/sro_minitaur/mpop_ic \
    --source {SRC} --target {TGT} --max-seq-len {MAXLEN}

### 3) Read the result

In [ ]:
import json
print(json.load(open(f'/content/drive/MyDrive/sro_minitaur/results/incontext/{SRC}_{TGT}.json')))